## 深度域子波动态增益模型

从 `wavelet_batch_synthetic_depth@20260427.ipynb` 的井旁合成记录 QC 产物中重新做动态切段，拟合 `ln(gain) ~ ln(seismic_rms)`，然后对整个深度域地震体预计算 dynamic gain volume。

该 gain model 是绝对 gain：后续训练配置中应与固定标量增益互斥使用，即 `fixed_gain=None` 且提供 `dynamic_gain_model_file`。


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import cigsegy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_seismic
from cup.seismic.survey import open_survey

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 120)


In [ ]:
data_root = repo_root / "data"

seismic_file = data_root / "raw" / "mero_84_coord_extend"
segy_options = {
    "iline": 5,
    "xline": 21,
    "istep": 1,
    "xstep": 4,
}
keylocs = [
    segy_options["iline"],
    segy_options["xline"],
    segy_options["istep"],
    segy_options["xstep"],
]

wavelet_batch_output_dir = data_root / "output_wavelet_batch_synthetic_depth_20260428"
batch_metrics_file = wavelet_batch_output_dir / "wavelet_batch_metrics.csv"

output_dir = data_root / "output_wavelet_dynamic_gain_model_depth_20260429"
figure_dir = output_dir / "figures"
training_samples_file = output_dir / "dynamic_gain_training_samples.csv"
fit_metrics_file = output_dir / "dynamic_gain_fit_metrics.csv"
gain_npz_file = output_dir / "dynamic_gain_depth.npz"
gain_segy_file = output_dir / "dynamic_gain_depth.segy"
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

# Adaptive segmentation parameters for well-side training samples.
min_segment_valid_samples = 8
max_segment_count = 20
min_segments_per_well = 1
gain_eps = 1e-12

selected_attribute = "seismic_rms"
min_batch_corr = None
application_rms_window_m = None
prediction_clip_percentiles = (5.0, 95.0)
attribute_floor_fraction = 0.10
volume_batch_traces = 2048

for path in [seismic_file, batch_metrics_file]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Seismic:", seismic_file)
print("Batch metrics:", batch_metrics_file)
print("Output dir:", output_dir)


In [ ]:
def resolve_artifact_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else repo_root / path


def centered_moving_sum(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    window_samples = int(max(window_samples, 1))
    if values.size == 0:
        return values.copy()
    radius_left = window_samples // 2
    radius_right = window_samples - 1 - radius_left
    padded = np.pad(values, (radius_left, radius_right), mode="constant", constant_values=0.0)
    cumsum = np.concatenate([[0.0], np.cumsum(padded)])
    return cumsum[window_samples:] - cumsum[:-window_samples]


def centered_moving_rms(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values)
    numerator = centered_moving_sum(np.where(valid, values**2, 0.0), window_samples)
    denominator = centered_moving_sum(valid.astype(float), window_samples)
    out = np.full(values.shape, np.nan, dtype=float)
    positive = denominator > 0.0
    out[positive] = np.sqrt(numerator[positive] / denominator[positive])
    return out


def centered_moving_sum_axis(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    window_samples = int(max(window_samples, 1))
    radius_left = window_samples // 2
    radius_right = window_samples - 1 - radius_left
    padded = np.pad(values, ((0, 0), (radius_left, radius_right)), mode="constant", constant_values=0.0)
    cumsum = np.concatenate(
        [np.zeros((values.shape[0], 1), dtype=np.float64), np.cumsum(padded, axis=1, dtype=np.float64)], axis=1
    )
    return cumsum[:, window_samples:] - cumsum[:, :-window_samples]


def centered_moving_rms_axis(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    valid = np.isfinite(values)
    numerator = centered_moving_sum_axis(np.where(valid, values**2, 0.0), window_samples)
    denominator = centered_moving_sum_axis(valid.astype(np.float32), window_samples)
    out = np.full(values.shape, np.nan, dtype=np.float32)
    positive = denominator > 0.0
    out[positive] = np.sqrt(numerator[positive] / denominator[positive]).astype(np.float32)
    return out


def zscore_traces(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    finite = np.isfinite(values)
    counts = finite.sum(axis=1, keepdims=True)
    if np.any(counts <= 0):
        raise ValueError("At least one trace has no finite samples.")
    means = np.where(finite, values, 0.0).sum(axis=1, keepdims=True) / counts
    centered = np.where(finite, values - means, 0.0)
    stds = np.sqrt((centered**2).sum(axis=1, keepdims=True) / counts)
    if np.any(~np.isfinite(stds) | (stds <= 0.0)):
        raise ValueError("At least one trace has non-positive standard deviation.")
    return centered / stds


def resolve_odd_window_samples(window_m: float, sample_step_m: float) -> int:
    window_samples = int(round(float(window_m) / float(sample_step_m)))
    window_samples = max(window_samples, 3)
    if window_samples % 2 == 0:
        window_samples += 1
    return window_samples


def finite_pearson(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    if np.nanstd(x[mask]) <= 0.0 or np.nanstd(y[mask]) <= 0.0:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])


def fit_log_linear(x_log: np.ndarray, y_log: np.ndarray) -> dict:
    x_log = np.asarray(x_log, dtype=float)
    y_log = np.asarray(y_log, dtype=float)
    mask = np.isfinite(x_log) & np.isfinite(y_log)
    if int(mask.sum()) < 5:
        raise ValueError("Need at least five finite samples for log-linear fitting.")
    x = x_log[mask]
    y = y_log[mask]
    design = np.column_stack([np.ones_like(x), x])
    intercept, slope = np.linalg.lstsq(design, y, rcond=None)[0]
    pred = intercept + slope * x
    ss_res = float(np.sum((y - pred) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot <= 0.0 else 1.0 - ss_res / ss_tot
    return {
        "intercept": float(intercept),
        "slope": float(slope),
        "n_samples": int(mask.sum()),
        "pearson": finite_pearson(x, y),
        "r2_in_sample": float(r2),
        "mae_log_gain": float(np.mean(np.abs(y - pred))),
        "rmse_log_gain": float(np.sqrt(np.mean((y - pred) ** 2))),
    }


def split_valid_indices(
    valid_indices: np.ndarray, *, min_valid_samples: int, max_segments: int, min_segments: int
) -> list[np.ndarray]:
    valid_indices = np.asarray(valid_indices, dtype=np.int64)
    if valid_indices.size < int(min_valid_samples):
        return []
    segment_count = int(min(max_segments, valid_indices.size // int(min_valid_samples)))
    segment_count = max(int(min_segments), segment_count)
    return [segment for segment in np.array_split(valid_indices, segment_count) if segment.size >= min_valid_samples]


def positive_ls_gain(
    seismic_values: np.ndarray, synthetic_raw_values: np.ndarray, *, eps: float, min_valid_samples: int
) -> float:
    seismic_values = np.asarray(seismic_values, dtype=float)
    synthetic_raw_values = np.asarray(synthetic_raw_values, dtype=float)
    valid = np.isfinite(seismic_values) & np.isfinite(synthetic_raw_values)
    if int(valid.sum()) < int(min_valid_samples):
        return np.nan
    numerator = float(np.sum(seismic_values[valid] * synthetic_raw_values[valid]))
    denominator = float(np.sum(synthetic_raw_values[valid] ** 2))
    gain = numerator / (denominator + float(eps) * max(int(valid.sum()), 1))
    return float(gain) if np.isfinite(gain) and gain > 0.0 else np.nan


def segment_attribute_values(seismic_values: np.ndarray) -> dict:
    seismic_values = np.asarray(seismic_values, dtype=float)
    finite = np.isfinite(seismic_values)
    if not np.any(finite):
        return {"seismic_rms": np.nan, "seismic_abs_mean": np.nan, "seismic_abs_p90": np.nan}
    values = seismic_values[finite]
    abs_values = np.abs(values)
    return {
        "seismic_rms": float(np.sqrt(np.nanmean(values**2))),
        "seismic_abs_mean": float(np.nanmean(abs_values)),
        "seismic_abs_p90": float(np.nanpercentile(abs_values, 90.0)),
    }


def set_log_column(df: pd.DataFrame, source_column: str, log_column: str) -> None:
    values = df[source_column].to_numpy(dtype=float)
    df[log_column] = np.nan
    positive = values > 0.0
    df.loc[positive, log_column] = np.log(values[positive])


def eval_mask_to_bool(series: pd.Series) -> np.ndarray:
    if series.dtype == bool:
        return series.to_numpy(dtype=bool)
    return series.astype(str).str.lower().isin(["true", "1", "yes"]).to_numpy(dtype=bool)


In [ ]:
survey = open_survey(seismic_file, seismic_type="segy", segy_options=segy_options)
geometry_depth = survey.query_geometry(domain="depth")
print(json.dumps(geometry_depth, indent=2, ensure_ascii=False))
assert geometry_depth["sample_domain"] == "depth"
assert geometry_depth["sample_unit"] == "m"
sample_step_m = float(geometry_depth["sample_step"])


In [ ]:
batch_metrics_df = pd.read_csv(batch_metrics_file)
required_metric_columns = {"well_name", "status", "scale", "synthetic_qc_path", "depth_shift_curve_path", "corr"}
missing = required_metric_columns - set(batch_metrics_df.columns)
if missing:
    raise ValueError(f"Batch metrics missing columns: {sorted(missing)}")

ok_batch_df = batch_metrics_df.loc[batch_metrics_df["status"] == "ok"].copy()
if min_batch_corr is not None:
    ok_batch_df = ok_batch_df.loc[ok_batch_df["corr"] >= float(min_batch_corr)].copy()
if ok_batch_df.empty:
    raise ValueError("No successful batch synthetic wells available for fitting.")

segment_rows = []
for _, row in ok_batch_df.iterrows():
    well_name = str(row["well_name"])
    scale = float(row["scale"])
    if not np.isfinite(scale) or scale <= 0.0:
        continue

    qc_path = resolve_artifact_path(row["synthetic_qc_path"])
    depth_shift_curve_path = resolve_artifact_path(row["depth_shift_curve_path"])
    if not qc_path.exists():
        raise FileNotFoundError(qc_path)
    if not depth_shift_curve_path.exists():
        raise FileNotFoundError(depth_shift_curve_path)

    qc_df = pd.read_csv(qc_path)
    required_qc_columns = {"twt_s", "seismic_norm", "synthetic_scaled", "eval_mask"}
    missing = required_qc_columns - set(qc_df.columns)
    if missing:
        raise ValueError(f"Synthetic QC {qc_path} missing columns: {sorted(missing)}")

    depth_shift_df = pd.read_csv(depth_shift_curve_path)
    required_depth_columns = {"twt_s", "tvdss_m"}
    missing = required_depth_columns - set(depth_shift_df.columns)
    if missing:
        raise ValueError(f"Depth shift curve {depth_shift_curve_path} missing columns: {sorted(missing)}")

    twt_s = qc_df["twt_s"].to_numpy(dtype=float)
    seismic_norm = qc_df["seismic_norm"].to_numpy(dtype=float)
    synthetic_scaled = qc_df["synthetic_scaled"].to_numpy(dtype=float)
    synthetic_raw = synthetic_scaled / scale
    eval_mask = eval_mask_to_bool(qc_df["eval_mask"])

    depth_twt = depth_shift_df["twt_s"].to_numpy(dtype=float)
    depth_z = depth_shift_df["tvdss_m"].to_numpy(dtype=float)
    finite_depth = np.isfinite(depth_twt) & np.isfinite(depth_z)
    if int(finite_depth.sum()) < 2:
        raise ValueError(f"Depth shift curve {depth_shift_curve_path} has too few finite samples.")
    order = np.argsort(depth_twt[finite_depth])
    depth_twt = depth_twt[finite_depth][order]
    depth_z = depth_z[finite_depth][order]

    finite = eval_mask & np.isfinite(twt_s) & np.isfinite(seismic_norm) & np.isfinite(synthetic_raw)
    segments = split_valid_indices(
        np.flatnonzero(finite),
        min_valid_samples=min_segment_valid_samples,
        max_segments=max_segment_count,
        min_segments=min_segments_per_well,
    )

    for segment_id, segment_indices in enumerate(segments):
        seg_twt = twt_s[segment_indices]
        seg_seismic = seismic_norm[segment_indices]
        seg_synthetic_raw = synthetic_raw[segment_indices]
        gain = positive_ls_gain(
            seg_seismic,
            seg_synthetic_raw,
            eps=gain_eps,
            min_valid_samples=min_segment_valid_samples,
        )
        if not np.isfinite(gain):
            continue
        tvdss_values = np.interp(seg_twt, depth_twt, depth_z)
        segment_rows.append(
            {
                "well_name": well_name,
                "segment_id": int(segment_id),
                "n_valid_samples": int(segment_indices.size),
                "twt_min_s": float(np.nanmin(seg_twt)),
                "twt_max_s": float(np.nanmax(seg_twt)),
                "tvdss_min_m": float(np.nanmin(tvdss_values)),
                "tvdss_max_m": float(np.nanmax(tvdss_values)),
                "segment_thickness_m": float(np.nanmax(tvdss_values) - np.nanmin(tvdss_values)),
                "gain": gain,
                "batch_scale": scale,
                "batch_corr": float(row["corr"]),
                **segment_attribute_values(seg_seismic),
            }
        )

training_samples_df = pd.DataFrame(segment_rows)
if training_samples_df.empty:
    raise ValueError("No finite training segments were generated from batch synthetic outputs.")
for column in ["gain", "seismic_rms", "seismic_abs_mean", "seismic_abs_p90"]:
    set_log_column(training_samples_df, column, f"log_{column}")
training_samples_df.to_csv(training_samples_file, index=False)

fit_df = training_samples_df.loc[
    np.isfinite(training_samples_df["log_gain"]) & np.isfinite(training_samples_df[f"log_{selected_attribute}"])
].copy()
if fit_df.empty:
    raise ValueError(f"No finite ln(gain) / ln({selected_attribute}) samples.")

fit = fit_log_linear(
    fit_df[f"log_{selected_attribute}"].to_numpy(dtype=float), fit_df["log_gain"].to_numpy(dtype=float)
)
log_gain_clip = tuple(np.nanpercentile(fit_df["log_gain"].to_numpy(dtype=float), prediction_clip_percentiles))
attribute_floor = float(
    np.nanpercentile(fit_df[selected_attribute].to_numpy(dtype=float), 1.0) * attribute_floor_fraction
)
attribute_floor = max(attribute_floor, np.finfo(float).tiny)

segment_thickness_m = fit_df["segment_thickness_m"].to_numpy(dtype=float)
segment_thickness_m = segment_thickness_m[np.isfinite(segment_thickness_m) & (segment_thickness_m > 0.0)]
application_window_m = (
    float(np.nanmedian(segment_thickness_m)) if application_rms_window_m is None else float(application_rms_window_m)
)
window_samples = resolve_odd_window_samples(application_window_m, sample_step_m)

fit_metrics_df = pd.DataFrame(
    [
        {
            **fit,
            "target": "log_gain",
            "attribute": f"log_{selected_attribute}",
            "n_training_segments": int(fit_df.shape[0]),
            "n_training_wells": int(fit_df["well_name"].nunique()),
            "log_gain_clip_p05": float(log_gain_clip[0]),
            "log_gain_clip_p95": float(log_gain_clip[1]),
            "gain_clip_p05": float(np.exp(log_gain_clip[0])),
            "gain_clip_p95": float(np.exp(log_gain_clip[1])),
            "attribute_floor": attribute_floor,
            "application_window_m": application_window_m,
            "application_window_samples": int(window_samples),
            "sample_step_m": sample_step_m,
            "gain_model_is_relative_to_fixed_gain": False,
            "intended_fixed_gain": None,
            "batch_metrics_file": str(batch_metrics_file),
            "training_samples_file": str(training_samples_file),
        }
    ]
)
fit_metrics_df.to_csv(fit_metrics_file, index=False)

print(f"ln(gain) = {fit['intercept']:.3f} + {fit['slope']:.3f} * ln({selected_attribute})")
print(
    "Training segments:",
    int(fit_metrics_df.loc[0, "n_training_segments"]),  # type: ignore
    "wells:",
    int(fit_metrics_df.loc[0, "n_training_wells"]),  # type: ignore
)
print("Gain clip:", float(fit_metrics_df.loc[0, "gain_clip_p05"]), "to", float(fit_metrics_df.loc[0, "gain_clip_p95"]))  # type: ignore
print("RMS window:", window_samples, "samples =", window_samples * sample_step_m, "m")
print("Saved", training_samples_file)
print("Saved", fit_metrics_file)
display(fit_metrics_df)


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 5.0), constrained_layout=True)
for well_name, well_df in fit_df.groupby("well_name"):
    ax.scatter(well_df[f"log_{selected_attribute}"], well_df["log_gain"], s=24, alpha=0.75, label=well_name)
x_min, x_max = np.nanpercentile(fit_df[f"log_{selected_attribute}"], [1, 99])
x_line = np.linspace(float(x_min), float(x_max), 100)
ax.plot(x_line, fit["intercept"] + fit["slope"] * x_line, color="black", lw=1.3, label="OLS fit")
ax.set_xlabel(f"ln({selected_attribute})")
ax.set_ylabel("ln(gain)")
ax.set_title("Dynamic gain training segments")
ax.grid(True, alpha=0.25)
ax.legend(loc="best", fontsize=7)
fit_fig = figure_dir / "qc_01_dynamic_gain_fit.png"
fig.savefig(fit_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", fit_fig)


In [ ]:
print("Loading seismic volume...")
seismic_volume = import_seismic(
    seismic_file,
    seismic_type="segy",
    iline=segy_options["iline"],
    xline=segy_options["xline"],
    istep=segy_options["istep"],
    xstep=segy_options["xstep"],
)
expected_shape = (int(geometry_depth["n_il"]), int(geometry_depth["n_xl"]), int(geometry_depth["n_sample"]))
if tuple(seismic_volume.shape) != expected_shape:
    raise ValueError(f"Seismic volume shape {seismic_volume.shape} != geometry shape {expected_shape}")

flat = seismic_volume.reshape(-1, seismic_volume.shape[-1])
gain_flat = np.empty_like(flat, dtype=np.float32)
for start in range(0, flat.shape[0], volume_batch_traces):
    end = min(start + volume_batch_traces, flat.shape[0])
    seismic_zscore = zscore_traces(flat[start:end])
    seismic_rms = centered_moving_rms_axis(seismic_zscore, window_samples)
    rms_safe = np.maximum(seismic_rms, float(attribute_floor))
    log_seismic_rms = np.where(np.isfinite(rms_safe) & (rms_safe > 0.0), np.log(rms_safe), np.nan)
    log_gain = float(fit["intercept"]) + float(fit["slope"]) * log_seismic_rms
    log_gain = np.clip(log_gain, float(log_gain_clip[0]), float(log_gain_clip[1]))
    gain_flat[start:end] = np.exp(log_gain).astype(np.float32)
    if start == 0 or end == flat.shape[0] or (start // volume_batch_traces) % 10 == 0:
        print(f"Processed traces {end}/{flat.shape[0]}")

gain_volume = gain_flat.reshape(seismic_volume.shape)
print("Gain volume shape:", gain_volume.shape)
print(
    "Gain min/median/max:",
    float(np.nanmin(gain_volume)),
    float(np.nanmedian(gain_volume)),
    float(np.nanmax(gain_volume)),
)


In [ ]:
ilines = np.arange(
    float(geometry_depth["inline_min"]),
    float(geometry_depth["inline_max"]) + 0.5 * float(geometry_depth["inline_step"]),
    float(geometry_depth["inline_step"]),
).astype(np.float32)
xlines = np.arange(
    float(geometry_depth["xline_min"]),
    float(geometry_depth["xline_max"]) + 0.5 * float(geometry_depth["xline_step"]),
    float(geometry_depth["xline_step"]),
).astype(np.float32)
samples = np.arange(
    float(geometry_depth["sample_min"]),
    float(geometry_depth["sample_max"]) + 0.5 * float(geometry_depth["sample_step"]),
    float(geometry_depth["sample_step"]),
).astype(np.float32)

metadata = {
    "artifact": gain_npz_file.name,
    "gain_model_type": "dynamic_rms_log_linear_depth",
    "gain_model_is_relative_to_fixed_gain": False,
    "intended_usage": "Set fixed_gain=None and dynamic_gain_model_file to this NPZ/SEG-Y model.",
    "selected_attribute": selected_attribute,
    "intercept": float(fit["intercept"]),
    "slope": float(fit["slope"]),
    "log_gain_clip": [float(log_gain_clip[0]), float(log_gain_clip[1])],
    "gain_clip": [float(np.exp(log_gain_clip[0])), float(np.exp(log_gain_clip[1]))],
    "attribute_floor": float(attribute_floor),
    "application_window_m": float(application_window_m),
    "application_window_samples": int(window_samples),
    "sample_step_m": sample_step_m,
    "min_segment_valid_samples": int(min_segment_valid_samples),
    "max_segment_count": int(max_segment_count),
    "min_segments_per_well": int(min_segments_per_well),
    "n_training_segments": int(fit_df.shape[0]),
    "n_training_wells": int(fit_df["well_name"].nunique()),
    "source_seismic_file": str(seismic_file),
    "batch_metrics_file": str(batch_metrics_file),
    "fit_metrics_file": str(fit_metrics_file),
}

np.savez_compressed(
    gain_npz_file,
    volume=gain_volume.astype(np.float32),
    ilines=ilines,
    xlines=xlines,
    samples=samples,
    geometry_json=json.dumps(geometry_depth, ensure_ascii=False),
    metadata_json=json.dumps(metadata, ensure_ascii=False),
)
print("Saved NPZ:", gain_npz_file)


In [ ]:
def build_textual_header(title: str, lines: list[str]) -> str:
    rows = [f"C{idx:>2d} {text}"[:80].ljust(80) for idx, text in enumerate([title, *lines], start=1)]
    rows.extend([f"C{idx:>2d}".ljust(80) for idx in range(len(rows) + 1, 41)])
    textual = "".join(rows)
    if len(textual) != 3200:
        raise ValueError(f"Expected 3200-char textual header, got {len(textual)}")
    return textual


textual = build_textual_header(
    "Depth-domain dynamic gain model from RMS attribute",
    [
        f"artifact={gain_npz_file.name}",
        "model=ln(gain)=intercept+slope*ln(seismic_rms)",
        f"intercept={fit['intercept']:.6g} slope={fit['slope']:.6g}",
        f"rms_window_m={application_window_m:.3f} samples={window_samples}",
        f"gain_clip={float(np.exp(log_gain_clip[0])):.6g}-{float(np.exp(log_gain_clip[1])):.6g}",
        "usage=fixed_gain None; dynamic_gain_model_file this artifact",
        "source=wavelet_batch_synthetic_depth_dynamic_segments",
    ],
)
cigsegy.create_by_sharing_header(
    str(gain_segy_file),
    str(seismic_file),
    np.ascontiguousarray(gain_volume.astype(np.float32)),
    keylocs=keylocs,
    textual=textual,
)
print("Saved SEG-Y:", gain_segy_file)


In [ ]:
mid_inline_idx = gain_volume.shape[0] // 2
mid_xline_idx = gain_volume.shape[1] // 2

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
axes[0].hist(gain_volume[np.isfinite(gain_volume)].ravel(), bins=80, color="tab:blue", alpha=0.85)
axes[0].set_xlabel("Gain")
axes[0].set_ylabel("Count")
axes[0].set_title("Dynamic gain distribution")
axes[0].grid(True, alpha=0.25)

im1 = axes[1].imshow(gain_volume[mid_inline_idx].T, aspect="auto", origin="upper", cmap="viridis")
axes[1].set_title(f"Inline index {mid_inline_idx}")
axes[1].set_xlabel("Xline index")
axes[1].set_ylabel("Depth sample")
fig.colorbar(im1, ax=axes[1], shrink=0.82)

im2 = axes[2].imshow(gain_volume[:, mid_xline_idx, :].T, aspect="auto", origin="upper", cmap="viridis")
axes[2].set_title(f"Xline index {mid_xline_idx}")
axes[2].set_xlabel("Inline index")
axes[2].set_ylabel("Depth sample")
fig.colorbar(im2, ax=axes[2], shrink=0.82)

gain_qc_fig = figure_dir / "qc_02_dynamic_gain_volume.png"
fig.savefig(gain_qc_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", gain_qc_fig)


In [ ]:
print("Dynamic gain model artifacts:")
print(gain_npz_file)
print(gain_segy_file)
print()
print("Fit/training artifacts:")
print(training_samples_file)
print(fit_metrics_file)
print()
print("Figures:")
for path in sorted(figure_dir.glob("*.png")):
    print(path)
